# Same Ask, Side by Side: Vector Search vs Vector Search + Metadata

> *"A family-friendly sports movie about a team, with three or more people in frame."*

Every poster is stored in ApertureDB with a CLIP embedding (what it *looks like*) and the object enrichments Google Cloud Storage produced
(labels, faces, content-safety ratings, object boxes). Left: the nearest posters by embedding alone. Right: the same search, but only posters
that satisfy the metadata filter may compete. The filter runs *inside* the vector search, so you still get `K` results.

| Part of the ask | Who knows it | How it enters the query |
|---|---|---|
| a sports movie **about a team** | the embedding (what it looks like) | the vector, ranking the candidates |
| a **sports** movie | the label detector | a graph hop: posters carrying a sports label |
| **three or more people** in frame | the face detector | a count: `face_count >= 3` (a vector cannot count) |
| **family-friendly** | the content-safety detector | adult and violent ratings no higher than UNLIKELY |

In [ ]:
import config
import vector_search as vs
from demo_view import fewer_than, show_query, side_by_side

client = config.get_client()
ASK = "a sports movie about a team"
K = 8
LABELS = vs.SPORT_LABELS                  # the label detector says "sport", "football", "basketball", ...
CRITERIA = vs.people_criteria(3)          # three or more faces, ratings no higher than UNLIKELY
DATASET = None                            # or "<bucket>/<prefix>" to search one ingested folder

In [ ]:
left = vs.search(client, ASK, k=K, dataset=DATASET)
right = vs.search(client, ASK, k=K, criteria=CRITERIA, label=LABELS, dataset=DATASET)
exact = [vs.exact_search(client, ASK, k=K, dataset=DATASET), vs.exact_search(client, ASK, k=K, criteria=CRITERIA, label=LABELS, dataset=DATASET)]
side_by_side(left, right, exact, checks=[fewer_than(3)])

*Reading the numbers:* **too few people** and **rated POSSIBLE+** are counted from the stored detector values. **recall@K** compares the index's answer
with an exact brute-force search over the same candidates (the index is approximate, so it is worth checking). Vectors are 512-dimensional CLIP ViT-B/16,
cosine similarity, HNSW index.

## The Query on the Right

In [ ]:
show_query(vs.nearest_posters(K, CRITERIA, dataset=DATASET, label=LABELS))

## Overlap: A Player Holding a Ball

Some questions are about how things sit *relative to each other*. A vector cannot say whether a person's box and a ball's box overlap; the database can
(`RegionIoU` over the object detector's boxes). Here only posters where a person overlaps a ball may compete.

In [ ]:
ASK2 = "a player holding a ball"
touching = vs.overlapping_ids(client, first="person|man|woman|boy|girl", second="ball", dataset=DATASET)
print(f"{len(touching)} posters have a person box overlapping a ball box")

no_ball = lambda row: None if row["image_id"] in set(touching) else "no ball on a person"
left2 = vs.search(client, ASK2, k=K, dataset=DATASET)
right2 = vs.search(client, ASK2, k=K, criteria=vs.only_ids(touching), dataset=DATASET)
exact2 = [vs.exact_search(client, ASK2, k=K, dataset=DATASET), vs.exact_search(client, ASK2, k=K, criteria=vs.only_ids(touching), dataset=DATASET)]
side_by_side(left2, right2, exact2, checks=[no_ball])

*Try your own ask:* change `ASK`, `LABELS` or the number of people and run the cells again (for example `"a team celebrating together"`).